# **A Data Science Agentic AI Team**


## Team Introduction

We are a Data Science Agentic AI Team that participates in Kaggle competitions. Our objective is to collaborate as a team of specialized AI agents to analyze datasets, engineer features, build predictive models, evaluate results, and generate valid Kaggle submissions.

Our team is composed of the following members:


* **Data Science Manager Agent**: The team leader whose job is to analyze the Kaggle challenge, create the execution plan and assign tasks to the other agents.
>* Deliverable: Data Science Plan
* **Problem Framing Agent**: examines the challenge/competition we are entering and decides what the team is trying to predict, what evaluation metric Kaggle uses, and want consititutes a valid/successful submission.
>* Deliverable: Problem Defintion, Evaluation Metric & Success Critieria
* **Data Analysis Agent**: Familiarizes itself with the dataset, examines it, identifies missing values, analyzes distributions, detects anomalies & highlights potentially useful predictors.
>* Deliverable: Data Analysis Report, Data Quality Assessment
* **Feature Engineering Agent**: examines the dataset and suggests and creates additional features that will be useful for enhancing the dataset so we can answer the question asked. It will also perform data cleaning, enocding, imputation & dataset preparation.
>* Deliverable: Feature Engineering Report, Training, Test and Validation Datasets
* **Modeling Agent**: Decides wha type of AI/ML model the team will use, creates the model, trains it and documents model decisions
>* Deliverable: Model Candidates, Tranining Results, Model Recommendations
* **Evaluation Agent**: takes the trained model, tests it against the test and validation datasets, approves or rejects model candidates and reports results. It compares performance, identifies weaknesses & determines if the model is ready for submission.
>* Deliverable: Evaluation report, Best Model Selection
* **Competition Agent**: receieves approved models, manages experiments, generates Kaggle submission files, tracks leaderboard performance, and recommends future improvements.
>* Deliverable: Submission File, Experiment Log & Improvement Recommendations

## Team Task

Our primary objective is to successfully participate in a Kaggle challenges. Given a dataseet & problem description, we will
* Understand the problem.
* Analyze the data.
* Engineer features.
* Train candidate models.
* Evaluate model performance.
* Generate a valid Kaggle submission.
* Iteratively improve leaderboard performance.


## Team Evaluation

We use langfuse for observability and evaluation

Metrics collected include:

* Token consumption
* Agent execution traces
* Tool selection correctness
* Tool call correctness
* Model performance metrics
* Validation scores
* Kaggle leaderboard scores

The ultimate measure of success is improvement in Kaggle leaderboard performance while maintaining a transparent and observable workflow.

##SYSTEM DESIGN

#Structured Output Schemas/Contracts between Agents

* Problem Framing Agent Output/Data Analysis Agent Input:
```
{
    "competition_name": "",
    "prediction_target": "",
    "evaluation_metric": "",
    "submission_columns": [],
    "success_criteria": "",
    "important_constraints": []
}
```
* Data Analysis Agent Output/Feature Engineering Agent Input:
```
{
    "dataset_shape": "",
    "target_variable": "",
    "missing_values": [],
    "candidate_predictors": [],
    "data_quality_issues": [],
    "key_findings": []
}
```
* Feature Engineering Agent Output/Modeling Engineering Agent Input:
```
{
    "competition_name": "",
    "new_features": [],
    "dropped_features": [],
    "encoding_strategy": "",
    "imputation_strategy": "",
    "feature_columns": [],
    "target_column": "",
    "train_shape_after_processing":[],
    "validation_shape":[],
    "test_shape_after_processing":[],
    "recommended_modeling_approach":[],
    "feature_rationale": []
}
```
* Modeling Engineer Agent Output/Evaluation Agent Input:
```
{
    "model_type": "",
    "hyperparameters": {},
    "training_score": "",
    "reason_for_selection": ""
}
```
* Evaluation Agent Output/Competition Agent Input:
```
{
    "model_type": "",
    "validation_score": "",
    "approved": True,
    "strengths": [],
    "weaknesses": [],
    "recommendations": []
}
```
* Competition Agent Output:
```
{
    "experiment_id": "",
    "model_type": "",
    "validation_score": "",
    "leaderboard_score": "",
    "submission_file": ""
}
```



In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
#install needed libs
!pip install --upgrade aisuite aisuite[anthropic] #Andrew Ng's wrapper lib that calls different LLM provides and abstracts each providers' API needs
!pip install anthropic #for calling anthropic LLMs through aisuite
!pint install kaggle
!pip install tavily-python
!pip install openai #for calling OpenAi's LLMs through aisuite
!pip install pypandoc
!pip install markdownify
!pip install tiktoken
!pip install -U "langfuse>2.0.0" #obervability & evals

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.9/863.9 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: docstring-parser
    Found existing installation: docstring_parser 0.18.0
    Uninstalling docstring_parser-0.18.0:
      Successfully uninstalled docstring_parser-0.18.0
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
/bin/bash: line 1: pint: command not foun

##Setup Langfuse

We do all the langfuse setup in one cell. Using the API directly to avoid a stale SDK client using stale values

In [2]:
from google.colab import userdata
from langfuse import Langfuse, get_client
from langfuse import observe

LANGFUSE_PUBLIC_KEY = userdata.get("LANGFUSE_PUBLIC_KEY_DS_TEAM").strip()
LANGFUSE_SECRET_KEY = userdata.get("LANGFUSE_SECRET_KEY_DS_TEAM").strip()
LANGFUSE_BASE_URL = "https://cloud.langfuse.com"

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    base_url=LANGFUSE_BASE_URL,
)

try:
    projects = langfuse.api.projects.get()
    print("✅ Connected. Projects:", [p.name for p in projects.data])
except Exception as e:
    print("❌ Langfuse connection failed")
    print("Type:", type(e).__name__)
    print("Message:", str(e))

✅ Connected. Projects: ['data-science-kaggle-team']


Next we import the required libs and setup a few tools the agents will have access to:





In [4]:
#import all the needed libs
import base64
import json
import os
import re
from datetime import datetime
from io import BytesIO
import pprint
import requests
import textwrap
import ast
import re
import json
import tiktoken
import pypandoc #to create the docx version of the final report


# 3rd Party
from IPython.display import display, Image as ImageDisplay, IFrame, Markdown
import aisuite as ai
import urllib, urllib.request
import urllib.parse # Import urllib.parse
import openai as _openai
from google.colab import files
from markdownify import markdownify

# libs needed to process datasets downloaded from kaggle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import zipfile
import pandas as pd
import numpy as np



## get all the api keys we are going to use
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAPI-API-KEY').strip()

##set up kaggle access
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')

!mkdir -p ~/.kaggle

kaggle_config = {
    "username": os.environ['KAGGLE_USERNAME'],
    "key": os.environ['KAGGLE_KEY']
}

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(kaggle_config, f)

# Set file permissions for safety (Read/Write only by owner)
!chmod 600 ~/.kaggle/kaggle.json

print("USERNAME:", os.environ.get("KAGGLE_USERNAME"))
print("KEY EXISTS:", os.environ.get("KAGGLE_KEY") is not None)
print("API_KEY EXISTS:", os.environ.get("KAGGLE_API_KEY") is not None)

#initialize ai client
Client = ai.Client()

#initialize langfuse client
langfuse_client = get_client()
_judge_client = _openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])

if langfuse_client.auth_check():
  print("✅ Langfuse connected successfully")
else:
  print("❌ Langfuse connection failed")


# We are going to use claude sonnet 4.5 for this team
#model="anthropic:claude-sonnet-4-5-20250929"

#using cheaper Anthropic models for most of the agents, except the editor
AGENT_MODEL_MAP = {
    "project_manager_agent": "anthropic:claude-haiku-4-5",
    "research_agent": "anthropic:claude-haiku-4-5",
    "corporate_strategy_manager_agent": "anthropic:claude-sonnet-4-6",
    "technical_writer_agent": "anthropic:claude-haiku-4-5",
    "editor_agent": "anthropic:claude-sonnet-4-6",
    "memory_summarizer_agent": "anthropic:claude-haiku-4-5",
}

DEFAULT_MODEL = "anthropic:claude-haiku-4-5"


AGENT_MAX_TOKENS = {
    "research_agent": 2000,
    "technical_writer_agent": 5000,
    "editor_agent": 8000,
    "memory_summarizer_agent": 700,
}


#slidesGPT API endpoint
slidesgpt_api_endpoint = "https://api.slidesgpt.com/v1/presentations/generate"

USERNAME: icarovazquez
KEY EXISTS: True
API_KEY EXISTS: False
✅ Langfuse connected successfully


##Kaggle Helper Section

This section contains a list of helper functions that deal with the datasets/files downloaded from Kaggle.


In [5]:
from pathlib import Path

KAGGLE_BASE_PATH = Path("/content/kaggle")

#get the Kaggle competition path
def get_competition_path(competition_name: str) -> Path:
  """
  Returns the local path to the competition directory.
  Example: titanic -> /content/kaggle/titanic
  """
  return KAGGLE_BASE_PATH / competition_name

#downloads competition's files
def download_competition_files(competition_name: str) -> Path:
  """
  Downloads and unzips the competition files to the competition directory.
  Example: titanic -> /content/kaggle/titanic/train.csv
  """
  competition_path = get_competition_path(competition_name)
  competition_path.mkdir(parents=True, exist_ok=True)

  !kaggle competitions download -c {competition_name} -p {str(competition_path)}

  for zip in competition_path.glob("*.zip"):
    with zipfile.ZipFile(zip, 'r') as zip_ref:
      zip_ref.extractall(competition_path)

  return competition_path

#list the competition files that were downloaded
def list_competition_files(competition_name: str):
  """
  Returns a list of all the files in the competition directory.
  Example: titanic -> ['train.csv', 'test.csv']
  """
  competition_path = get_competition_path(competition_name)

  if not competition_path.exists():
    raise FileNotFoundError(
        f"No local foluder for Competition {competition_name}"
        "Run download_competition_files() first"
    )

  return [p.name for p in competition_path.iterdir()]


#load the competition files in pandas dataframes
def load_competition_data(
    competition_name: str,
    train_file: str = "train.csv",
    test_file: str = "test.csv",
    sample_submission_file: str = "gender_submission.csv",
):
  """
  Loads test, train & sample submission files for a Kaggle competition.
  """
  competition_path = get_competition_path(competition_name)

  if not competition_path.exists():
    raise FileNotFoundError(
        f"No local folder for Competition {competition_name}."
        "Run download_competition_files() first"
    )

  train_path = competition_path / train_file
  test_path = competition_path / test_file
  sample_submission_path = competition_path / sample_submission_file

  missing = [
      str(path)
      for path in [train_path, test_path, sample_submission_path]
      if not path.exists()
  ]

  if missing:
    raise FileNotFoundError(
         f"Missing expected competition files:\n{chr(10).join(missing)}\nRun download_competition_files() first."
    )

  train_df = pd.read_csv(train_path)
  test_df = pd.read_csv(test_path)
  sample_submission_df = pd.read_csv(sample_submission_path)

  return train_df, test_df, sample_submission_df


  #create the file that will be submitted to kaggle
  def create_submission_file(
      ids,
      predictions,
      id_column_name: str = "PassengerId",
      prediction_column_name: str = "Survived",
      output_path: str = "submission.csv",
  ):

    """
    Creates a Kaggle submission file.
    Default values work for the Titanic competition.
    """

    submission_df = pd.DataFrame({
        id_column_name: ids,
        prediction_column_name: predictions,
    })

    submission_df.to_csv(output_path, index=False)

    return output_oath, submission_df


  #submits the submission file to Kaggle
  def submit_to_kaggle(
      competition_name: str,
      submission_file: str,
      message: str,
  ):

    """
    Submits a Kaggle submission file to a competition.
    """

    !kaggle competitions submit -c {competition_name} -f {submission_file} -m "{message}"



In [6]:
#testing if our helper functions work

competition_name="titanic"

download_competition_files("titanic")

print(list_competition_files(competition_name))


train_df, test_df, sample_submission_df = load_competition_data(competition_name)

print("training dataset shape is: ", train_df.shape)
print("test dataset shape is: ", test_df.shape)
print("sample submission file shape is: ", sample_submission_df.shape)

print(train_df.head())
print(test_df.head())
print(sample_submission_df.head())

100% 34.1k/34.1k [00:00<00:00, 6.75MB/s]

['gender_submission.csv', 'titanic.zip', 'train.csv', 'test.csv']
training dataset shape is:  (891, 12)
test dataset shape is:  (418, 11)
sample submission file shape is:  (418, 2)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN   

Temporary Initialization of the team's history

In [7]:
history= []

def add_to_history(history, step, agent_name, result):
  history.append((step, agent_name, result))

##Instrumented LLM call

Before defining the agents we need to create a reusuable LLM call that instruments langfuse. All agents will use this function

In [8]:
@observe(name="llm_call", as_type='generation')
def llm_call(agent_name:str, messages: list, temperature: float = 1.0, tools: list= None) -> str:

  """
  observability wrapper that all agents will use when calling the llm
  logs model, input, output, and token usage to Langfuse
  """

  selected_model = AGENT_MODEL_MAP.get(agent_name, DEFAULT_MODEL)

  max_tokens = AGENT_MAX_TOKENS.get(agent_name, 3000)

  messages = trim_messages(messages, max_chars=12000)

  call_kwargs = {
    "model": selected_model,
    "messages": messages,
    "temperature": temperature,
    "max_tokens": max_tokens,
  }

  if tools:
    call_kwargs["tools"] = tools

  #calling the llm
  response = Client.chat.completions.create(**call_kwargs)
  content = response.choices[0].message.content

  #log output after calling the LLM
  usage = getattr(response, "usage", None)

  input_tokens = getattr(usage, "prompt_tokens", 0) if usage else 0
  output_tokens = getattr(usage, "completion_tokens", 0) if usage else 0
  total_tokens = input_tokens + output_tokens

  result = {
        "agent_name": agent_name,
        "model": selected_model,
        "temperature": temperature,
        "content": content,
        "usage": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens
        } if usage else {},
        "tools_available": bool(tools),
    }

  print('llm call completed')

  return result


# Helper Function to Trim Amount of Data Sent to LLMs

This helper function decides how much context to send to the LLMs

In [9]:
def trim_messages(messages, max_chars=12000):

    trimmed = []
    total_chars = 0

    # start from newest messages first
    for msg in reversed(messages):

        content = msg.get("content", "")

        if not isinstance(content, str):
            content = str(content)

        remaining = max_chars - total_chars

        if remaining <= 0:
            break

        # trim oversized message if needed
        if len(content) > remaining:
            content = content[-remaining:]

        trimmed.append({
            **msg,
            "content": content
        })

        total_chars += len(content)

    # restore chronological order
    return list(reversed(trimmed))


# Helper Function to Count Token Usage
This helper function will summarize token usage

In [10]:
def summarize_token_usage(history):
    total_input = 0
    total_output = 0
    total_tokens = 0

    for step, agent, output in history:
        if isinstance(output, dict):
            usage = output.get("usage", {})
            total_input += usage.get("input_tokens", 0)
            total_output += usage.get("output_tokens", 0)
            total_tokens += usage.get("total_tokens", 0)

    return {
        "input_tokens": total_input,
        "output_tokens": total_output,
        "total_tokens": total_tokens,
    }

## **Helper Function that cleans LLM call Output**

This helper function cleans the raw output returned by the LLM call

In [11]:
def clean_llm_dict_output(raw_output: str) -> str:
    cleaned = raw_output.strip()

    if cleaned.startswith("```"):
        lines = cleaned.splitlines()

        # Remove opening fence, e.g. ```python or ```json
        if lines[0].strip().startswith("```"):
            lines = lines[1:]

        # Remove closing fence
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned = "\n".join(lines).strip()

    return cleaned

## **Helper Function that Tracks how Our Agentic Workflow is performing**

Finally, we add a helper function that keeps track of some metrics that measure how our Agentic workflow is performing

In [12]:
def summarize_run_metrics(result):

  history = result.get("history", [])

  agent_metrics = {}

  total_input_tokens = 0
  total_output_tokens = 0
  total_tokens = 0

  for step, agent_name, output in history:
    if agent_name not in agent_metrics:
      agent_metrics[agent_name] = {
          "calls": 0,
          "input_tokens": 0,
          "output_tokens": 0,
          "total_tokens": 0,
      }

    agent_metrics[agent_name]["calls"] +=1

    if isinstance(output, dict):
      usage = output.get("usage", {})

      input_tokens= usage.get("input_tokens", 0)
      output_tokens = usage.get("output_tokens", 0)
      step_total_tokens = usage.get("total_tokens", input_tokens+output_tokens)

      total_input_tokens += input_tokens
      total_output_tokens += output_tokens
      total_tokens += step_total_tokens

      agent_metrics[agent_name]["input_tokens"] += input_tokens
      agent_metrics[agent_name]["output_tokens"] += output_tokens
      agent_metrics[agent_name]["total_tokens"] += step_total_tokens

  report = result.get("report", "")


  return {
      "total_input_tokens": total_input_tokens,
      "total_output_tokens": total_output_tokens,
      "total_tokens": total_tokens,
      "report_chars": len(report),
      "report_words": len(report.split()),
      "scores": result.get("scores", {}),
      "agent_metrics": agent_metrics,
  }


## **Problem Framing Agent**

We start by creating our first agent. The Problem Framing Agent examines the downloaded competition files, examines them and returns info on what the team is going to try to predict

In [13]:
@observe(name="problem_framing_agent")
def problem_framing_agent(competition_name: str):

  print("==================================")
  print("Problem Framing Agent")
  print("==================================")

  train_df, test_df, sample_submission_df = load_competition_data(competition_name)

  prompt = f"""
  You are a problem framing agent for a Data Science Agentic AI Team.
  Your job is to to understand the Kaggle competition setup and return a structured problem definition

  Use the available information

  Competition Name: {competition_name}

  Training Columns: {list(train_df.columns)}

  Test Columns: {list(test_df.columns)}

  Sample Submission Columns: {list(sample_submission_df.columns)}

  Training Shape: {train_df.shape}

  Test Shape: {test_df.shape}

  Sample Submission Preview: {sample_submission_df.head().to_string(index=False)}

  Return ONLY a valid Python dictionary with the following structure:

  {{
    "competition_name": "",
    "problem_type": "",
    "prediction_target": "",
    "evaluation_metric": "",
    "submission_columns": [],
    "success_criteria": "",
    "important_constraints": []
  }}

  Rules:
  - Do not write markdown
  - Do not explain your reasoning
  - Return only the dictionary
  """

  messages = [{"role": "user","content":prompt}]

  llm_result = llm_call(
      agent_name="problem_framing_agent",
      messages=messages,
      temperature=0.2,
  )

  raw_output = llm_result["content"]

  cleaned_output = clean_llm_dict_output(raw_output)

  try:
    problem_definition = ast.literal_eval(cleaned_output)
  except Exception as e:
    print("Could not parse structured problem framing output:", e)
    problem_definition = {
        "competition_name": competition_name,
        "problem_type": "",
        "prediction_target": "",
        "evaluation_metric": "",
        "submission_columns": list(sample_submission_df.columns),
        "success_criteria": "Generate a valid Kaggle submission file",
        "important_constraints": [f"Parsing failed: {str(e)}"],
        "raw_output": raw_output
    }

  result = {
    **llm_result,
    "problem_definition": problem_definition
  }

  print(problem_definition)

  add_to_history(
      history,
      "Frame Titanic Competition",
      "problem_framing_agent",
      result
  )

  return result


In [14]:
problem_result = problem_framing_agent("titanic")

print(problem_result['problem_definition'])

Problem Framing Agent
llm call completed
{'competition_name': 'titanic', 'problem_type': 'binary_classification', 'prediction_target': 'Survived', 'evaluation_metric': 'accuracy', 'submission_columns': ['PassengerId', 'Survived'], 'success_criteria': 'Maximize accuracy of survival predictions on test set', 'important_constraints': ['Target variable only present in training data', 'Test set has 418 samples without survival labels', 'Training set has 891 samples', 'Predictions must be binary (0 or 1)', 'PassengerId must match test set PassengerId in submission', 'Handle missing values in Age, Cabin, and Embarked columns']}
{'competition_name': 'titanic', 'problem_type': 'binary_classification', 'prediction_target': 'Survived', 'evaluation_metric': 'accuracy', 'submission_columns': ['PassengerId', 'Survived'], 'success_criteria': 'Maximize accuracy of survival predictions on test set', 'important_constraints': ['Target variable only present in training data', 'Test set has 418 samples wit


## **Data Analysis Agent**
The second agent we add is the Data Analysis Agent. This agent receives the output from the Problem Framing Agent and creates a Data Report

In [15]:
@observe(name="data_analysis_agent")
def data_analysis_agent(competition_name: str, problem_definition: dict):

  print("==================================")
  print("Data Analysis Agent")
  print("==================================")

  train_df, test_df, sample_submission_df = load_competition_data(competition_name)

  target_variable = problem_definition["prediction_target"]

  dataset_summary = {
      "train_shape": train_df.shape,
      "test_shape": test_df.shape,
      "train_columns": list(train_df.columns),
      "test_columns": list(test_df.columns),
      "sample_submission_columns": list(sample_submission_df.columns),
      "train_dtype": train_df.dtypes.astype(str).to_dict(),
      "test_dtype": test_df.dtypes.astype(str).to_dict(),
      "missing_values_train": train_df.isnull().sum().to_dict(),
      "missing_values_test": test_df.isnull().sum().to_dict(),
  }

  if target_variable in train_df.columns:
    dataset_summary["target_distribution"] = train_df[target_variable].value_counts(normalize=True).round(4).to_dict()
  else:
    dataset_summary["target_distribution"] = "Target variable not found in dataset"

  numerical_features = train_df.select_dtypes(include=['int64','float64']).columns.tolist()
  categorical_features = train_df.select_dtypes(include=["object","category"]).columns.tolist()

  dataset_summary['numerical_features'] = numerical_features
  dataset_summary['categorical_features'] = categorical_features

  prompt = f"""
  You are a data analysis agent for a Data Science Agentic AI Team.

  Your job is to pefrorm exploratory data analysis on the given Kaggle dataset and return a structured Data Analysis record.

  Competition name: {competition_name}

  Problem definition: {problem_definition}

  Dataset summary: {dataset_summary}

  Return ONLY a valid Python dictionary with the following structure:

  {{
    "competition_name": "",
    "dataset_shape": {{}},
    "target_variable": "",
    "target_distribution": {{}},
    "missing_values":{{}},
    "numerical_features": [],
    "categorical_features": [],
    "candidate_predictors": []
    "data_quality_issues": [],
    "key_finidings": [],
    "recommended_next_steps": []
  }}

  Represent shapes as lists, not tuples.
  Example:
  "train": [891, 12]

  Rules:
  - Do not write markdown
  - Do not explain your reasoning
  - Return only the dictionary
  - Focus on practical insights that help feature engineering and modeling.

  """

  messages = [{"role": "user","content":prompt}]

  llm_result = llm_call(
      agent_name="data_analysis_agent",
      messages=messages,
      temperature=0.2,
  )

  raw_output = llm_result["content"]

  cleaned_output = clean_llm_dict_output(raw_output)

  try:
    data_analysis_record = ast.literal_eval(cleaned_output)
  except Exception as e:
    print("Could not parse structured data analysis output:", e)
    data_analysis_record = {
        "competition_name": competition_name,
        "dataset_shape": {
            "train_shape": train_df.shape,
            "test_shape": test_df.shape,
        },
        "target_variable": target_variable,
        "target_distribution": dataset_summary["target_distribution"],
        "missing_values": {
            "train": dataset_summary["missing_values_train"],
            "test": dataset_summary["missing_values_test"]
        },
        "numerical_features": dataset_summary["numerical_features"],
        "categorical_features": dataset_summary["categorical_features"],
        "candidate_predictors": [],
        "data_quality_issues": [f"Parsing failed: {str(e)}"],
        "key_findings": [],
        "recommended_next_steps": [],
        "raw_output": raw_output

    }


  result = {
    **llm_result,
    "data_analysis_record": data_analysis_record
  }

  print(data_analysis_record)

  add_to_history(
      history,
      "Analyze Competition Data",
      "data_analysis_agent",
      result
  )

  return result



In [16]:
problem_definition = problem_result["problem_definition"]

data_analysis_result = data_analysis_agent("titanic", problem_definition)

data_analysis_result["data_analysis_record"]


Data Analysis Agent
llm call completed
{'competition_name': 'titanic', 'dataset_shape': {'train': [891, 12], 'test': [418, 11]}, 'target_variable': 'Survived', 'target_distribution': {'class_0': 0.6162, 'class_1': 0.3838, 'imbalance_ratio': 1.61}, 'missing_values': {'train': {'Age': 177, 'Cabin': 687, 'Embarked': 2}, 'test': {'Age': 86, 'Cabin': 327, 'Fare': 1}}, 'numerical_features': ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], 'categorical_features': ['Sex', 'Cabin', 'Embarked', 'Name', 'Ticket'], 'candidate_predictors': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Cabin'], 'data_quality_issues': ['Age has 19.9% missing values in train and 20.6% in test', 'Cabin has 77.1% missing values in train and 78.3% in test', 'Embarked has 0.2% missing values in train', 'Fare has 0.2% missing values in test', 'Target variable imbalance with 61.6% negative class and 38.4% positive class', 'Name and Ticket columns have high cardinality and require feature engineering'], 'key_fi

{'competition_name': 'titanic',
 'dataset_shape': {'train': [891, 12], 'test': [418, 11]},
 'target_variable': 'Survived',
 'target_distribution': {'class_0': 0.6162,
  'class_1': 0.3838,
  'imbalance_ratio': 1.61},
 'missing_values': {'train': {'Age': 177, 'Cabin': 687, 'Embarked': 2},
  'test': {'Age': 86, 'Cabin': 327, 'Fare': 1}},
 'numerical_features': ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'],
 'categorical_features': ['Sex', 'Cabin', 'Embarked', 'Name', 'Ticket'],
 'candidate_predictors': ['Pclass',
  'Sex',
  'Age',
  'SibSp',
  'Parch',
  'Fare',
  'Embarked',
  'Cabin'],
 'data_quality_issues': ['Age has 19.9% missing values in train and 20.6% in test',
  'Cabin has 77.1% missing values in train and 78.3% in test',
  'Embarked has 0.2% missing values in train',
  'Fare has 0.2% missing values in test',
  'Target variable imbalance with 61.6% negative class and 38.4% positive class',
  'Name and Ticket columns have high cardinality and require feature engineering'],
 'key_fi